<img src="https://i.postimg.cc/J7sf3Lq4/Group.png" alt="Logo" width="220" />

# Sesión 3 — NumPy (90 minutos) | Guion del profesor

## Propósito de la sesión
En 1h 30m buscamos que el grupo **entienda y practique** los pilares de NumPy que más se usan en ciencia de datos e ingeniería:
- `ndarray` (forma, tipo, memoria)
- indexación (slicing y máscaras)
- vectorización + broadcasting
- agregaciones por `axis`
- reshape/transpose/stack
- álgebra lineal aplicada (resolver sistemas)
- random reproducible y mini simulación

## Plan sugerido (90 min)
- **0–8 min**: Setup + motivación + `ndarray`
- **8–22 min**: Creación + atributos + `dtype`
- **22–42 min**: Indexación/slicing/máscaras (la parte más importante)
- **42–58 min**: Vectorización + broadcasting
- **58–70 min**: Agregaciones por `axis` + `where/clip`
- **70–82 min**: Reshape/transpose/concatenate/stack
- **82–90 min**: Álgebra lineal (solve) + mini Monte Carlo (cierre)

## Reglas de dinámica
- Cada bloque tiene una **pregunta de control** para el grupo.
- Evitar loops en Python salvo para comparación.


In [2]:
import numpy as np

rng = np.random.default_rng(12345)
np.set_printoptions(precision=4, suppress=True)
np.__version__

'2.2.6'

## 1) ¿Por qué NumPy?

Puntos a transmitir (rápido):
- Arreglos homogéneos: mejor para CPU/cache.
- Operaciones vectorizadas: más rápidas, más legibles.
- Base de pandas, scikit-learn, etc.

**Control al grupo:** ¿qué significa que un array sea “homogéneo”?

## 2) `ndarray` — forma, dimensiones, tipo

Idea clave: la mayoría de problemas en NumPy son problemas de **forma (`shape`)**.


In [3]:
A = np.array([[0, 1, 2, 3],
              [4, 5, 6, 7],
              [1, 2, 3, 4]])

print(A)
print("shape:", A.shape)
print("ndim:", A.ndim)
print("size:", A.size)
print("dtype:", A.dtype)

[[0 1 2 3]
 [4 5 6 7]
 [1 2 3 4]]
shape: (3, 4)
ndim: 2
size: 12
dtype: int64


### `dtype` y conversión

En vivo conviene mostrar:
- cómo cambia el tipo
- por qué `float` aparece cuando hay división
- `astype` como herramienta estándar

**Control al grupo:** ¿qué pasa con el dtype si mezclo enteros y floats?

In [4]:
v_int = np.array([1, 2, 3, 4])
v_float = v_int / 3
v_cast = v_float.astype(np.int32)

v_int.dtype, v_float.dtype, v_cast.dtype, v_float, v_cast

(dtype('int64'),
 dtype('float64'),
 dtype('int32'),
 array([0.3333, 0.6667, 1.    , 1.3333]),
 array([0, 0, 1, 1], dtype=int32))

## 3) Creación de arrays (lo que más se usa)

- `arange`: paso fijo
- `linspace`: n puntos
- especiales: `zeros`, `ones`, `eye`, `full`

**Control al grupo:** ¿cuándo conviene `linspace` en vez de `arange`?

In [5]:
v1 = np.arange(0, 10, 2)
v2 = np.linspace(0, 1, 6)

Z = np.zeros((2, 3))
O = np.ones((2, 3))
I = np.eye(4)
F = np.full((2, 3), 7)

v1, v2, Z, O, I, F

(array([0, 2, 4, 6, 8]),
 array([0. , 0.2, 0.4, 0.6, 0.8, 1. ]),
 array([[0., 0., 0.],
        [0., 0., 0.]]),
 array([[1., 1., 1.],
        [1., 1., 1.]]),
 array([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]]),
 array([[7, 7, 7],
        [7, 7, 7]]))

## 4) Random reproducible (forma moderna)

Recomendación: `default_rng`.

**Control al grupo:** si vuelvo a correr la celda, ¿obtengo lo mismo? ¿por qué?

In [6]:
u = rng.random(5)
n = rng.standard_normal(5)
e = rng.integers(0, 100, size=10)
u, n, e

(array([0.2273, 0.3168, 0.7974, 0.6763, 0.3911]),
 array([-0.7409, -1.3678,  0.6489,  0.3611, -1.9529]),
 array([70, 24, 91, 94, 73, 66, 13,  9, 26, 44]))

## 5) Indexación y slicing (donde se aprende NumPy de verdad)

Objetivos de este bloque:
- leer y cortar
- extraer filas/columnas
- usar máscaras para filtrar/modificar
- entender vista vs copia (sin meternos a bajo nivel)

**Control al grupo:** ¿qué devuelve `M[:, 1]`?

In [7]:
M = np.array([[ 0,  1,  2,  3],
              [ 4,  5,  6,  7],
              [ 8,  9, 10, 11],
              [ 3,  5,  6,  9]])
M

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [ 3,  5,  6,  9]])

In [8]:
fila_0 = M[0]
col_1 = M[:, 1]
bloque = M[1:3, 1:3]

print("fila_0:", fila_0)
print("col_1:", col_1)
print("bloque:\n", bloque)

fila_0: [0 1 2 3]
col_1: [1 5 9 5]
bloque:
 [[ 5  6]
 [ 9 10]]


### Máscaras booleanas (filtrar y modificar)

**Control al grupo:** ¿qué tipo de objeto es `mask` y qué forma tiene?

In [9]:
mask = M >= 6
print("mask shape:", mask.shape)
print("cantidad True:", mask.sum())
print("valores filtrados:", M[mask])

mask shape: (4, 4)
cantidad True: 8
valores filtrados: [ 6  7  8  9 10 11  6  9]


In [10]:
M_mod = M.copy()
M_mod[M_mod < 5] = 0
M_mod

array([[ 0,  0,  0,  0],
       [ 0,  5,  6,  7],
       [ 8,  9, 10, 11],
       [ 0,  5,  6,  9]])

### `where` para condicionales

Muy útil cuando queremos construir un nuevo array sin modificar el original.

**Control al grupo:** ¿`where` modifica `M` o crea uno nuevo?

In [11]:
M_flag = np.where(M >= 6, 1, 0)
M_flag

array([[0, 0, 0, 0],
       [0, 0, 1, 1],
       [1, 1, 1, 1],
       [0, 0, 1, 1]])

### Vistas vs copias (demostración corta)

Sin teoría pesada: lo importante es entender que un slice puede compartir memoria.

**Control al grupo:** si modifico `sub`, ¿se modifica `M_view`?

In [12]:
M_view = M.copy()
sub = M_view[:2, :2]
sub[:] = 999
M_view

array([[999, 999,   2,   3],
       [999, 999,   6,   7],
       [  8,   9,  10,  11],
       [  3,   5,   6,   9]])

## 6) Vectorización vs loops (comparación conceptual)

Mensaje: “si tu primer solución tiene `for` anidados, probablemente NumPy puede hacerlo mejor”.

**Control al grupo:** ¿qué ventaja tiene vectorizar además de performance?

In [13]:
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])

x + y, x * y, x.dot(y)

(array([5, 7, 9]), array([ 4, 10, 18]), np.int64(32))

## 7) Broadcasting (lo que habilita la magia)

Ejemplo clásico: centrar y escalar columnas.

Puntos didácticos:
- `axis=0` para estadísticos por columna
- restar un vector `(p,)` a una matriz `(n,p)` funciona por broadcasting

**Control al grupo:** si `X` es (n, p), ¿qué forma tiene `X.mean(axis=0)`?

In [16]:
X = rng.integers(0, 100, size=(6, 4))
mu = X.mean(axis=0)
sigma = X.std(axis=0)

Xc = X - mu
Xs = (X - mu) / sigma

print("X shape:", X.shape)
print("mu shape:", mu.shape)
print("sigma shape:", sigma.shape)
print("media de Xs (axis=0):", Xs.mean(axis=0))
print("std  de Xs (axis=0):", Xs.std(axis=0))
X
mu

X shape: (6, 4)
mu shape: (4,)
sigma shape: (4,)
media de Xs (axis=0): [-0.  0. -0.  0.]
std  de Xs (axis=0): [1. 1. 1. 1.]


array([44.8333, 52.1667, 49.8333, 26.    ])

## 8) Agregaciones, `axis` y utilidades

Señalar:
- `sum/mean/std/min/max`
- `argmax/argmin` + `unravel_index`
- `clip` para recortar valores

**Control al grupo:** `axis=1` agrega por filas o por columnas?

In [17]:
Y = rng.integers(-50, 150, size=(5, 6))
Y

array([[ 72, -37,  58,  73, -44, -15],
       [ 28,  10, 140,  38,  79, -20],
       [111,  -7, 103,  44,  31,  45],
       [ 57,   1, -26,   9, 147,   5],
       [ 83,   2, 117,  46,  63,  -8]])

In [19]:
print("sum axis=0:", Y.sum(axis=0))
print("sum axis=1:", Y.sum(axis=1))

pos = np.unravel_index(Y.argmax(), Y.shape)
print("max global:", Y[pos], "en", pos)

Y_clip = np.clip(Y, 0, 100)
Y_clip

sum axis=0: [351 -31 392 210 276   7]
sum axis=1: [107 275 327 193 303]
max global: 147 en (np.int64(3), np.int64(4))


array([[ 72,   0,  58,  73,   0,   0],
       [ 28,  10, 100,  38,  79,   0],
       [100,   0, 100,  44,  31,  45],
       [ 57,   1,   0,   9, 100,   5],
       [ 83,   2, 100,  46,  63,   0]])

## 9) Reestructuración (reshape / transpose / concatenate / stack)

Meta: que el grupo no se pierda con shapes.

**Control al grupo:** si tengo 24 elementos, ¿qué shapes son posibles para `reshape`?

In [20]:
v = np.arange(24)
R = v.reshape(6, 4)
Rt = R.T

print("R shape:", R.shape)
print("Rt shape:", Rt.shape)
R

R shape: (6, 4)
Rt shape: (4, 6)


array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15],
       [16, 17, 18, 19],
       [20, 21, 22, 23]])

In [21]:
top = np.ones((2, 4))
bottom = np.zeros((2, 4))

cat0 = np.concatenate([top, bottom], axis=0)
cat1 = np.concatenate([top, bottom], axis=1)

print("concat axis=0 shape:", cat0.shape)
print(cat0)
print("concat axis=1 shape:", cat1.shape)
print(cat1)

concat axis=0 shape: (4, 4)
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
concat axis=1 shape: (2, 8)
[[1. 1. 1. 1. 0. 0. 0. 0.]
 [1. 1. 1. 1. 0. 0. 0. 0.]]


In [25]:
S0 = np.stack([R, R + 100], axis=0)
S1 = np.stack([R, R + 100], axis=1)

print("stack axis=0 shape:", S0.shape)
print(S0)
print("stack axis=1 shape:", S1.shape)
print(S1)

stack axis=0 shape: (2, 6, 4)
[[[  0   1   2   3]
  [  4   5   6   7]
  [  8   9  10  11]
  [ 12  13  14  15]
  [ 16  17  18  19]
  [ 20  21  22  23]]

 [[100 101 102 103]
  [104 105 106 107]
  [108 109 110 111]
  [112 113 114 115]
  [116 117 118 119]
  [120 121 122 123]]]
stack axis=1 shape: (6, 2, 4)
[[[  0   1   2   3]
  [100 101 102 103]]

 [[  4   5   6   7]
  [104 105 106 107]]

 [[  8   9  10  11]
  [108 109 110 111]]

 [[ 12  13  14  15]
  [112 113 114 115]]

 [[ 16  17  18  19]
  [116 117 118 119]]

 [[ 20  21  22  23]
  [120 121 122 123]]]


## 10) Álgebra lineal aplicada (lo imprescindible)

Mensaje principal:
- Para `Ax=b`, usar `np.linalg.solve(A, b)`.
- Evitar enseñar `inv(A) @ b` como práctica estándar.

**Control al grupo:** ¿qué forma tiene `A` para que el sistema tenga solución única?

In [ ]:
A = np.array([[2, 3, 3],
              [4, 1, 2],
              [1, 6, 1]])
b = np.array([10, 11, 9])

x = np.linalg.solve(A, b)
x

In [ ]:
np.allclose(A @ x, b), A @ x

## 11) Mini demo final: Monte Carlo para estimar π

Cierre práctico con:
- generación de puntos
- vectorización
- máscara
- promedio como probabilidad

**Control al grupo:** ¿por qué multiplicamos por 4?

In [ ]:
N = 150_000
pts = rng.uniform(-1, 1, size=(N, 2))
r2 = pts[:, 0]**2 + pts[:, 1]**2
inside = r2 <= 1
pi_hat = 4 * inside.mean()
pi_hat, np.pi, abs(pi_hat - np.pi)

## 12) Mini-reto oral (2–3 min)

Propuesta para cerrar:
1. Generar una matriz `D` (1000, 5) normal.
2. Estandarizar columnas.
3. Encontrar el valor absoluto máximo y reportar su posición.

Si hay tiempo, se ejecuta la celda siguiente.


In [ ]:
D = rng.standard_normal((1000, 5))
mu = D.mean(axis=0)
sigma = D.std(axis=0)
Dz = (D - mu) / sigma

idx = np.unravel_index(np.abs(Dz).argmax(), Dz.shape)
value = Dz[idx]

print("media por columna:", Dz.mean(axis=0))
print("std  por columna:", Dz.std(axis=0))
print("máximo |valor| en:", idx)
print("valor:", value)

## 13) Errores típicos (para recordar)

1. **Confundir `axis`** (0=columnas, 1=filas).
2. **Shape mismatch** en broadcasting (si no alinea, falla).
3. Usar `inv(A) @ b` como primera opción en vez de `solve(A, b)`.
4. Modificar un slice sin querer (vistas).

Cierre: “Si controlas `shape`, controlas NumPy.”
